# Part 2

In [1]:
import torch

In [2]:
import numpy as np
import pandas as pd

In [3]:
from conllu import parse_incr
import re

chunk_types = set()

for path in [
    "/srv/data/lt2326-h25/a2/hi_hdtb-ud-train.conllu",
    "/srv/data/lt2326-h25/a2/hi_hdtb-ud-test.conllu",
]:
    with open(path, "r", encoding="utf-8") as f:
        for tokenlist in parse_incr(f):
            for token in tokenlist:
                misc = token.get("misc")
                if misc and "ChunkId" in misc:
                    chunk_type = re.match(r"[A-Z]+", misc["ChunkId"]).group(0)
                    chunk_types.add(chunk_type)

In [4]:
device = torch.device('cuda:3')

* https://pypi.org/project/conllu/

In [5]:
label_list = ["O"]
for chunk in sorted(chunk_types):
    label_list.append(f"B-{chunk}")
    label_list.append(f"I-{chunk}")

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

In [6]:
label2id

{'O': 0,
 'B-BLK': 1,
 'I-BLK': 2,
 'B-CCP': 3,
 'I-CCP': 4,
 'B-FRAGP': 5,
 'I-FRAGP': 6,
 'B-JJP': 7,
 'I-JJP': 8,
 'B-NEGP': 9,
 'I-NEGP': 10,
 'B-NP': 11,
 'I-NP': 12,
 'B-RBP': 13,
 'I-RBP': 14,
 'B-VGF': 15,
 'I-VGF': 16,
 'B-VGNF': 17,
 'I-VGNF': 18,
 'B-VGNN': 19,
 'I-VGNN': 20}

In [7]:
len(label2id)

21

In [8]:
def read_hindi_for_bert(path, label2id):
    sentences = []

    with open(path, "r", encoding="utf-8") as f:
        for tokenlist in parse_incr(f):
            tokens = []
            labels = []

            prev_chunk = None

            for token in tokenlist:
                if token["form"] is None:
                    continue

                tokens.append(token["form"])

                misc = token.get("misc", {})
                chunk_id = misc.get("ChunkId")
                chunk_type = re.match(r"[A-Z]+", chunk_id).group(0)

                if chunk is None:
                    labels.append(label2id["O"])
                    prev_chunk = None
                else:
                    if chunk != prev_chunk:
                        labels.append(label2id[f"B-{chunk_type}"])
                    else:
                        labels.append(label2id[f"I-{chunk_type}"])
                    prev_chunk = chunk_type

            sentences.append({
                "tokens": tokens,
                "chunk_tags": labels
            })

    return sentences

In [9]:
train_sentences = read_hindi_for_bert('/srv/data/lt2326-h25/a2/hi_hdtb-ud-train.conllu', label2id)

In [10]:
train_sentences[0]

{'tokens': ['यह',
  'एशिया',
  'की',
  'सबसे',
  'बड़ी',
  'मस्जिदों',
  'में',
  'से',
  'एक',
  'है',
  '।'],
 'chunk_tags': [11, 11, 11, 11, 11, 11, 11, 11, 11, 15, 1]}

In [11]:
test_sentences = read_hindi_for_bert('/srv/data/lt2326-h25/a2/hi_hdtb-ud-test.conllu', label2id)

In [12]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"tokens": [x['tokens'] for x in train_sentences], "chunk_tags": [x["chunk_tags"] for x in train_sentences]})
test_dataset = Dataset.from_dict({"tokens": [x['tokens'] for x in test_sentences], "chunk_tags": [x["chunk_tags"] for x in test_sentences]})

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-multilingual-cased")

In [14]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [15]:
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/13306 [00:00<?, ? examples/s]

Map:   0%|          | 0/1684 [00:00<?, ? examples/s]

In [16]:
tokenized_train[0]

{'tokens': ['यह',
  'एशिया',
  'की',
  'सबसे',
  'बड़ी',
  'मस्जिदों',
  'में',
  'से',
  'एक',
  'है',
  '।'],
 'chunk_tags': [11, 11, 11, 11, 11, 11, 11, 11, 11, 15, 1],
 'input_ids': [101,
  13525,
  860,
  87084,
  10826,
  28603,
  76923,
  889,
  13432,
  73649,
  12878,
  15552,
  11497,
  10532,
  11072,
  11186,
  10569,
  920,
  102],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100,
  11,
  11,
  -100,
  11,
  11,
  11,
  11,
  -100,
  -100,
  -100,
  -100,
  -100,
  11,
  11,
  11,
  15,
  1,
  -100]}

In [17]:
from transformers import AutoModelForTokenClassification

model_name = "distilbert/distilbert-base-multilingual-cased"

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [19]:
import evaluate, seqeval

seqeval = evaluate.load("seqeval")

In [20]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [21]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./hindi-tag",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.121700,0.093326,0.976955,0.976418,0.976686,0.975303
2,0.074400,0.059311,0.985723,0.984896,0.985309,0.984420
3,0.049700,0.049762,0.988278,0.988021,0.988149,0.987327
4,0.040900,0.044819,0.989609,0.989265,0.989437,0.988795
5,0.032500,0.042125,0.990131,0.989902,0.990016,0.989500
6,0.029600,0.040865,0.990421,0.990249,0.990335,0.989839
7,0.018900,0.041019,0.990480,0.990422,0.990451,0.989783
8,0.020000,0.040636,0.990653,0.990567,0.990610,0.990037
9,0.018100,0.040048,0.990857,0.990885,0.990871,0.990432
10,0.016500,0.041473,0.990421,0.990307,0.990364,0.989867


/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/seqeval/m

TrainOutput(global_step=2080, training_loss=0.07364873079439768, metrics={'train_runtime': 1239.0175, 'train_samples_per_second': 107.392, 'train_steps_per_second': 1.679, 'total_flos': 3520988910915444.0, 'train_loss': 0.07364873079439768, 'epoch': 10.0})